<a href="https://colab.research.google.com/github/dora-lai/fyp-aspect-financial-news/blob/main/Tier_A_FiQA2018_Sentiment_Benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install packages
!pip install -q datasets transformers torch nltk scikit-learn pysentiment2 seaborn matplotlib

In [ ]:
# Import and setup
import os
import pandas as pd
import numpy as np
import torch
import nltk
import pysentiment2 as ps
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from datasets import load_dataset
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_recall_fscore_support, confusion_matrix
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import seaborn as sns
import matplotlib.pyplot as plt

DEVICE = 0 if torch.cuda.is_available() else -1
LABELS = ['positive', 'neutral', 'negative']
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

print(f"[INFO] Device: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")
print(f"[INFO] PyTorch version: {torch.__version__}")

[INFO] Device: CPU
[INFO] PyTorch version: 2.11.0+cpu


In [ ]:
# Load FiQA2018
print("\n[INFO] Loading FiQA 2018 dataset...")
dataset = load_dataset("pauri32/fiqa-2018")
print("[INFO] Dataset splits:", list(dataset.keys()))

df_test = pd.DataFrame(dataset['test'])
print("[INFO] Test columns:", list(df_test.columns))
print("[INFO] Test size:", len(df_test))

if 'sentence' not in df_test.columns:
    raise KeyError(f"Expected 'sentence' column. Found: {list(df_test.columns)}")

df_test['sentence'] = df_test['sentence'].astype(str)


[INFO] Loading FiQA 2018 dataset...


[INFO] Dataset splits: ['train', 'validation', 'test']
[INFO] Test columns: ['sentence', 'snippets', 'target', 'sentiment_score', 'aspects', 'format', 'label']
[INFO] Test size: 150


In [ ]:
# Build true_label
print("\n[INFO] Deriving true_label from sentiment_score thresholds.")
print(df_test['sentiment_score'].describe())

def convert_score_to_polarity(score, pos_th=0.1, neg_th=-0.1):
    if score > pos_th:
        return 'positive'
    elif score < neg_th:
        return 'negative'
    else:
        return 'neutral'

df_test['true_label'] = df_test['sentiment_score'].apply(convert_score_to_polarity)

print("\n[INFO] true_label distribution:")
print(df_test['true_label'].value_counts())

print("\n[INFO] Sample rows:")
print(df_test[['sentence', 'sentiment_score', 'true_label']].head(3).to_string())


[INFO] Deriving true_label from sentiment_score thresholds.
count    150.000000
mean      -0.003173
std        0.340927
min       -0.938000
25%       -0.292750
50%        0.000000
75%        0.250250
max        0.721000
Name: sentiment_score, dtype: float64

[INFO] true_label distribution:
true_label
positive    50
negative    50
neutral     50
Name: count, dtype: int64

[INFO] Sample rows:
                                                                                                          sentence  sentiment_score true_label
0                                $HCP Come to the party and buy this -gonna give solid gains and a dividend $$$$$$            0.520   positive
1  @gakrum nice chart shows distinctive down channel not a dip.. where do you see the bottom? $SPY ..$150? ..$130?           -0.443   negative
2                               Japan's Asahi to submit bid next week for SABMiller's Grolsch and Peroni - Yomiuri            0.236   positive


In [ ]:
# Task 13: Dataset structure counts + cut-off record
# ------------------------------------------------------------------------------
# Step 2b (Tracker Task 13): Dataset structure counts and cut-off record
# ------------------------------------------------------------------------------
print("\n[INFO] Tracker Task 13 — FiQA 2018 structure")

# 1. Headlines vs posts
if 'format' in df_test.columns:
    format_counts = df_test['format'].value_counts()
    print("\n[INFO] Headline vs post counts:")
    print(format_counts)
    format_counts.to_csv("results/tables/fiqa_format_counts.csv")
else:
    print("[WARN] No 'format' column found.")

# 2. Aspect category counts
if 'aspects' in df_test.columns:
    aspect_counts = df_test['aspects'].explode().value_counts()
    print("\n[INFO] Aspect category counts:")
    print(aspect_counts)
    aspect_counts.to_csv("results/tables/fiqa_aspect_counts.csv")
else:
    print("[WARN] No 'aspects' column found.")

# 3. Record cut-offs explicitly
cutoff_note = pd.DataFrame([{
    'task': 'Task 13',
    'dataset': 'FiQA 2018',
    'split': 'test',
    'positive_threshold': 0.1,
    'negative_threshold': -0.1,
    'rule': 'positive if score > 0.1; negative if score < -0.1; else neutral',
    'n_records': len(df_test),
}])
cutoff_note.to_csv("results/tables/fiqa_cutoff_record.csv", index=False)
print("\n[INFO] Saved cutoff record to results/tables/fiqa_cutoff_record.csv")


[INFO] Tracker Task 13 — FiQA 2018 structure

[INFO] Headline vs post counts:
format
post        81
headline    69
Name: count, dtype: int64

[INFO] Aspect category counts:
aspects
['Corporate/Appointment']                                              13
['Stock/Price Action']                                                 13
['Stock/Price Action/Bullish']                                         12
['Stock/Technical Analysis']                                           10
['Corporate/Dividend Policy']                                           9
['Stock/Price Action/Bearish/Bearish Behavior']                         9
['Stock/Price Action/Bullish/Bullish Behavior']                         6
['Corporate/Sales']                                                     5
['Stock/Price Action/Bearish']                                          5
['Corporate/Appointment/Executive Appointment']                         4
['Corporate/M&A/M&A']                                                   4
['Co

In [ ]:
# VADER baseline
print("\n[INFO] Running Baseline 1: VADER...")
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def evaluate_vader(text):
    compound = sia.polarity_scores(text)['compound']
    if compound >= 0.05:
        return 'positive'
    elif compound <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df_test['pred_vader'] = df_test['sentence'].apply(evaluate_vader)


[INFO] Running Baseline 1: VADER...


In [ ]:
# Loughran-McDonald baseline
print("\n[INFO] Running Baseline 2: Loughran-McDonald...")
lm = ps.LM()

def evaluate_lm(text):
    tokens = lm.tokenize(text)
    score = lm.get_score(tokens)
    polarity = score.get('Polarity', 0.0)
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'

df_test['pred_lm'] = df_test['sentence'].apply(evaluate_lm)


[INFO] Running Baseline 2: Loughran-McDonald...


In [ ]:
# FinBERT baseline
print("\n[INFO] Running Baseline 3: FinBERT (ProsusAI)...")
model_name = "ProsusAI/finbert"
print(f"[INFO] Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

finbert_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    truncation=True,
    max_length=512,
    batch_size=32,
)

finbert_preds = finbert_pipeline(df_test['sentence'].tolist())
df_test['pred_finbert'] = [p['label'].lower() for p in finbert_preds]


[INFO] Running Baseline 3: FinBERT (ProsusAI)...
[INFO] Loading model: ProsusAI/finbert


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Task 19: Reusable scoring harness
# ------------------------------------------------------------------------------
# Step 6a (Tracker Task 19): Reusable scoring harness
# ------------------------------------------------------------------------------
def evaluate_predictions(y_true, y_pred, labels=None, model_name="model"):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    if labels is None:
        labels = ['positive', 'neutral', 'negative']

    acc = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average='macro', zero_division=0
    )
    precision_w, recall_w, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average='weighted', zero_division=0
    )

    return {
        'Model': model_name,
        'Accuracy': round(acc, 4),
        'Precision (Macro)': round(precision_macro, 4),
        'Recall (Macro)': round(recall_macro, 4),
        'Macro F1': round(f1_macro, 4),
        'Weighted F1': round(f1_weighted, 4),
    }

In [ ]:
# Evaluation loop using the harness
print("\n" + "=" * 66)
print("           TIER A (FiQA 2018) — DOCUMENT-LEVEL BASELINES          ")
print("=" * 66)

methods = [
    ('VADER', 'pred_vader'),
    ('Loughran-McDonald', 'pred_lm'),
    ('FinBERT (ProsusAI)', 'pred_finbert'),
]

results = []
for name, col in methods:
    row = evaluate_predictions(
        df_test['true_label'], df_test[col],
        labels=LABELS, model_name=name
    )
    results.append(row)

    print(f"\n[DETAIL] {name} Classification Report:")
    print(classification_report(
        df_test['true_label'], df_test[col],
        labels=LABELS, digits=4, zero_division=0
    ))

comparison_table = pd.DataFrame(results)
print("\n[SUMMARY] Baseline comparison:")
print(comparison_table.to_string(index=False))

comparison_table.to_csv("results/tables/tier_a_baseline_metrics.csv", index=False)
df_test.to_csv("results/tables/tier_a_test_predictions.csv", index=False)
print("\n[INFO] Saved tier_a_baseline_metrics.csv")
print("[INFO] Saved tier_a_test_predictions.csv")


           TIER A (FiQA 2018) — DOCUMENT-LEVEL BASELINES          

[DETAIL] VADER Classification Report:
              precision    recall  f1-score   support

    positive     0.4167    0.5000    0.4545        50
     neutral     0.4211    0.4800    0.4486        50
    negative     0.4545    0.3000    0.3614        50

    accuracy                         0.4267       150
   macro avg     0.4308    0.4267    0.4215       150
weighted avg     0.4308    0.4267    0.4215       150


[DETAIL] Loughran-McDonald Classification Report:
              precision    recall  f1-score   support

    positive     0.2500    0.0800    0.1212        50
     neutral     0.3736    0.6800    0.4823        50
    negative     0.5116    0.4400    0.4731        50

    accuracy                         0.4000       150
   macro avg     0.3784    0.4000    0.3589       150
weighted avg     0.3784    0.4000    0.3589       150


[DETAIL] FinBERT (ProsusAI) Classification Report:
              precision    r

In [ ]:
# Confusion matrices
for name, col in methods:
    cm = confusion_matrix(df_test['true_label'], df_test[col], labels=LABELS)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in LABELS],
        columns=[f"pred_{l}" for l in LABELS]
    )
    print(f"\n[CONFUSION] {name}:")
    print(cm_df.to_string())

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    safe_name = name.replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(f"results/figures/confusion_{safe_name}.png", dpi=150)
    plt.close()

print("\n[INFO] Saved confusion matrices to results/figures/")


[CONFUSION] VADER:
               pred_positive  pred_neutral  pred_negative
true_positive             25            16              9
true_neutral              17            24              9
true_negative             18            17             15

[CONFUSION] Loughran-McDonald:
               pred_positive  pred_neutral  pred_negative
true_positive              4            37              9
true_neutral               4            34             12
true_negative              8            20             22

[CONFUSION] FinBERT (ProsusAI):
               pred_positive  pred_neutral  pred_negative
true_positive             19            25              6
true_neutral               6            31             13
true_negative              3            12             35

[INFO] Saved confusion matrices to results/figures/
